In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from air_quality_monitor.analysis import AirQualityAnalyser
from air_quality_monitor.storage import CSVStorage
from air_quality_monitor.models import AirQualityReading, WeatherReading
from air_quality_monitor.config import Config
from air_quality_monitor.logger import setup_logging
import logging
from pathlib import Path
import pandas as pd
from datetime import datetime


In [ ]:
LOG_LEVEL = "DEBUG"
setup_logging(log_level=LOG_LEVEL, log_dir=Config.LOG_DIR)
logger = logging.getLogger("mlnb")

In [ ]:
notebook_dir = Path().resolve()

aqi_filepath = notebook_dir.parent / "data" / "aqi_history.csv"
aqi_storage = CSVStorage(aqi_filepath, AirQualityReading)
aqi_df = aqi_storage.read()

we_filepath = notebook_dir.parent / "data" / "we_history.csv"
we_storage = CSVStorage(we_filepath, WeatherReading)
we_df  = we_storage.read()

print(aqi_df.dtypes)
print(we_df.dtypes)

In [ ]:
we_df['dt_floor'] = we_df['dt'].dt.floor('h')
print(we_df[['dt', 'dt_floor']])
aqi_df['dt_floor'] = aqi_df['pollutant_timestamp'].dt.floor('h')
print(aqi_df[['pollutant_timestamp', 'dt_floor']])

In [ ]:
aqi_df.drop(['latitude', 'longitude'], axis=1, inplace=True)
we_df.drop(['state', 'country', 'timezone', 'latitude', 'longitude'], axis=1, inplace=True)
print(we_df['dt_floor'].dtype)
print(aqi_df['dt_floor'].dtype)
we_df

In [ ]:
#full_df = aqi_df.join(we_df, on='dt_floor', how='outer', lsuffix='_aqi', rsuffix='_we')
full_df = pd.merge(aqi_df, we_df, how='outer', on=['city', 'dt_floor'], suffixes=("_aqi", "_we"))
full_df

In [ ]:
print(f"full_df:\t{full_df.shape}")
print(f"aqi_df:\t{aqi_df.shape}")
print(f"we_df:\t{we_df.shape}\n\n")
print(f"full_df NaNs:\t{full_df.isnull().sum()}")
print(f"aqi_df NaNs:\t{aqi_df.isnull().sum()}")
print(f"we_df NaNs:\t{we_df.isnull().sum()}")
